<a href="https://colab.research.google.com/github/khalid-alajmi/bayan-applied-nlp-khalid-alajmi/blob/main/notebooks/01_text_processing_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayan — Applied NLP
## Day 1 — Text Processing & Tokenisation
**Student:** Khalid Alajmi


In [1]:
# Bayan Applied NLP — Day 1: Text Processing & Tokenisation
# Student: Khalid Alajmi

# 1) Reproducible environment
import importlib.util
import subprocess
import sys

PINNED = {"transformers": "5.15.1", "tokenizers": "0.22.2", "spacy": "3.8.7"}
missing = [name for name in PINNED if importlib.util.find_spec(name) is None]
if missing:
    packages = [f"{name}=={PINNED[name]}" for name in missing]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# 2) Imports and synthetic bilingual data
import html
import re
import time
import unicodedata
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import spacy
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing

SEED = 42
rng = np.random.default_rng(SEED)

samples = [
    "طلب تجديد التصريح في الرياض ما زال قيد المعالجة.",
    "الخدمة ممتازة\u00a0وسريعة، شكراً لكم!",
    "For support, contact user@example.invalid",
    "رقم التواصل التجريبي: 0558764321",
    "The service request was completed successfully.",
]

print("Python:", sys.version.split()[0])
print("Synthetic samples:", len(samples))

# 3) Unicode inspection
def inspect_unicode(text: str) -> list[dict[str, str]]:
    return [
        {"char": ch, "code_point": f"U+{ord(ch):04X}", "name": unicodedata.name(ch, "UNKNOWN")}
        for ch in text
    ]

unicode_rows = inspect_unicode("أé")
assert unicode_rows[0]["code_point"] == "U+0623"
print("Unicode inspection=PASS")

# 4) Conservative preprocessing, two-copy contract, and PII masking
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"
WHITESPACE = re.compile(r"\s+")
HTML_TAG = re.compile(r"<[^>]+>")
EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
SAUDI_MOBILE = re.compile(r"(?<!\d)(?:\+?966|00966|0)?5\d{8}(?!\d)")

@dataclass(frozen=True)
class TextRecord:
    raw_text: str
    model_text: str


def mask_pii(text: str) -> str:
    return SAUDI_MOBILE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))


def prepare_text(text: str, *, remove_diacritics: bool = False, normalize_alef: bool = False) -> TextRecord:
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    raw = text
    model = unicodedata.normalize("NFC", html.unescape(text))
    model = HTML_TAG.sub(" ", model).replace(TATWEEL, "")
    if remove_diacritics:
        model = ARABIC_DIACRITICS.sub("", model)
    if normalize_alef:
        model = re.sub(r"[إأآٱ]", "ا", model)
    model = WHITESPACE.sub(" ", mask_pii(model)).strip()
    return TextRecord(raw_text=raw, model_text=model)

records = [prepare_text(text) for text in samples]
assert records[1].raw_text != records[1].model_text
assert "user@example.invalid" not in records[2].model_text
assert "0558764321" not in records[3].model_text
assert records[2].raw_text == samples[2]
print("Two-copy preprocessing contract=PASS")

# 5) Testable sentence segmentation on the protected model copy
sentence_nlp = spacy.blank("xx")
sentence_nlp.add_pipe("sentencizer")


def split_model_sentences(text: str) -> list[str]:
    protected = prepare_text(text).model_text
    return [s.text.strip() for s in sentence_nlp(protected).sents if s.text.strip()]

sentence_examples = {
    "ar": "تم تقديم الطلب بنجاح. ما زلت أنتظر التحديث.",
    "en": "The request was received. I am waiting for an update.",
}
sentence_splits = {lang: split_model_sentences(text) for lang, text in sentence_examples.items()}
abbreviation_probe = split_model_sentences("راجع د. أحمد. ثم أعد المحاولة.")
assert sentence_splits["ar"] == ["تم تقديم الطلب بنجاح.", "ما زلت أنتظر التحديث."]
assert sentence_splits["en"] == ["The request was received.", "I am waiting for an update."]
print("Known abbreviation probe:", abbreviation_probe)
print("SPACY_SENTENCE_PIPELINE=PASS")

# 6) Local educational WordPiece tokenizer
special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
known_tokens = [
    "طلب", "تجديد", "التصريح", "في", "الرياض", "ما", "زال", "قيد", "المعالجة", ".",
    "شكراً", "لكم", "الخدمة", "ممتازة", "وسريعة", "!", "For", "support", ",", "contact",
    "<", "EMAIL", ">", "رقم", "التواصل", "التجريبي", "The", "service", "request", "was",
    "completed", "successfully",
]
vocab_tokens = special_tokens + known_tokens
vocab = {token: idx for idx, token in enumerate(vocab_tokens)}
local_tokenizer = Tokenizer(WordPiece(vocab=vocab, unk_token="[UNK]"))
local_tokenizer.pre_tokenizer = BertPreTokenizer()
local_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[("[CLS]", vocab["[CLS]"]), ("[SEP]", vocab["[SEP]"])],
)
assert local_tokenizer.encode("طلب تجديد").tokens == ["[CLS]", "طلب", "تجديد", "[SEP]"]
print("Local WordPiece demonstration=PASS")

# 7) Token fertility and truncation metrics
def word_count(text: str) -> int:
    return max(1, len(text.split()))


def token_fertility(tokenizer: Tokenizer, text: str) -> float:
    tokens = tokenizer.encode(text).tokens
    content = [t for t in tokens if t not in {"[CLS]", "[SEP]", "[PAD]"}]
    return len(content) / word_count(text)


def truncation_rate(tokenizer: Tokenizer, texts: Iterable[str], max_length: int) -> float:
    texts = list(texts)
    if not texts:
        raise ValueError("texts must not be empty")
    return sum(len(tokenizer.encode(t).ids) > max_length for t in texts) / len(texts)

model_texts = [record.model_text for record in records]
fertilities = [token_fertility(local_tokenizer, text) for text in model_texts]
rate_at_10 = truncation_rate(local_tokenizer, model_texts, max_length=10)
mean_local_fertility = float(np.mean(fertilities))
print("Local mean token fertility:", round(mean_local_fertility, 2))
print("Local truncation rate @10:", f"{rate_at_10:.0%}")
assert all(x > 0 for x in fertilities)
assert 0.0 <= rate_at_10 <= 1.0
print("Tokenisation metrics=PASS")

# 8) Padding, truncation, attention mask, and simple embeddings
MAX_LENGTH = 12
local_tokenizer.enable_truncation(max_length=MAX_LENGTH)
local_tokenizer.enable_padding(length=MAX_LENGTH, pad_id=vocab["[PAD]"], pad_token="[PAD]")
batch = [local_tokenizer.encode(text) for text in model_texts[:2]]
input_ids = np.array([item.ids for item in batch], dtype=np.int64)
attention_mask = np.array([item.attention_mask for item in batch], dtype=np.int64)
embedding_table = rng.normal(0.0, 0.02, size=(len(vocab), 8))
embeddings = embedding_table[input_ids]
assert input_ids.shape == (2, MAX_LENGTH)
assert attention_mask.shape == input_ids.shape
assert embeddings.shape == (2, MAX_LENGTH, 8)
print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("embeddings shape:", embeddings.shape)
print("IDs-to-embeddings pipeline=PASS")

# Restore raw local-tokenizer behaviour before comparisons.
local_tokenizer.no_padding()
local_tokenizer.no_truncation()

# 9) Explore + Distinction: compare multilingual tokenizers on five synthetic samples
explore_samples = [
    "طلب تجديد التصريح ما زال قيد المعالجة.",
    "الخدمة ممتازة وتم إنجاز الطلب بسرعة.",
    "لم أستلم أي تحديث عن طلبي حتى الآن.",
    "The service request was completed successfully.",
    "I am still waiting for an update on my request.",
]

comparison_results = {}
try:
    from transformers import AutoTokenizer

    hf_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased", use_fast=True)
    xlmr_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base", use_fast=True)

    print("\n=== Explore: Local tokenizer vs mBERT ===")
    for i, text in enumerate(explore_samples, 1):
        prepared = prepare_text(text).model_text
        local_tokens = local_tokenizer.encode(prepared).tokens
        mbert_tokens = hf_tokenizer.tokenize(prepared)
        print(f"Sample {i}: local={len(local_tokens)} tokens, mBERT={len(mbert_tokens)} tokens, local_UNK={local_tokens.count('[UNK]')}")
    print("EXPLORE_TOKENIZER_COMPARISON=PASS")

    def hf_fertility(tokenizer, text: str) -> float:
        return len(tokenizer.tokenize(text)) / max(len(text.split()), 1)

    def truncation_rate_hf(tokenizer, texts, max_length=10) -> float:
        return sum(
            len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"]) > max_length
            for text in texts
        ) / len(texts)

    def measure_time(tokenizer, texts, repeats=100) -> float:
        start = time.perf_counter()
        for _ in range(repeats):
            for text in texts:
                tokenizer.tokenize(text)
        return (time.perf_counter() - start) / (repeats * len(texts))

    prepared_explore = [prepare_text(text).model_text for text in explore_samples]
    for name, tokenizer in {"mBERT": hf_tokenizer, "XLM-R": xlmr_tokenizer}.items():
        mean_fertility = float(np.mean([hf_fertility(tokenizer, text) for text in prepared_explore]))
        trunc_rate = truncation_rate_hf(tokenizer, prepared_explore, max_length=10)
        avg_ms = measure_time(tokenizer, prepared_explore) * 1000
        comparison_results[name] = {
            "mean_fertility": mean_fertility,
            "truncation_rate_at_10": trunc_rate,
            "avg_tokenization_ms": avg_ms,
        }
        print(f"{name}: fertility={mean_fertility:.2f}, truncation@10={trunc_rate:.0%}, avg_time={avg_ms:.3f} ms")
    print("DISTINCTION_TOKENIZER_COMPARISON=PASS")
except Exception as exc:
    print("HF tokenizer comparison unavailable in this run:", type(exc).__name__, str(exc)[:160])
    print("Core remains independently testable; rerun with network access for Explore/Distinction evidence.")

# 10) Core verification
core_checks = {
    "unicode": unicode_rows[0]["code_point"] == "U+0623",
    "spacy_sentence_pipeline": all(len(value) == 2 for value in sentence_splits.values()),
    "raw_copy_preserved": records[2].raw_text == samples[2],
    "pii_masked": "<EMAIL>" in records[2].model_text and "<PHONE>" in records[3].model_text,
    "token_metrics": all(x > 0 for x in fertilities) and 0 <= rate_at_10 <= 1,
    "embedding_shape": embeddings.shape == (2, MAX_LENGTH, 8),
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY1_NOTEBOOK1_CORE=PASS")

# 11) Compact evidence summary for README
print("\n=== DAY 1 EVIDENCE SUMMARY ===")
print("Profile: Conservative (NFC + whitespace normalization + tatweel removal + PII masking)")
print("PII example: 0558764321 -> <PHONE>")
print("Local mean token fertility:", round(mean_local_fertility, 2))
print("Local truncation rate @10:", f"{rate_at_10:.0%}")
print("Decision: Preserve Arabic text with conservative normalization; aggressive normalization is not used by default.")
if comparison_results:
    for name, result in comparison_results.items():
        print(f"{name}: fertility={result['mean_fertility']:.2f}, truncation@10={result['truncation_rate_at_10']:.0%}, avg_time={result['avg_tokenization_ms']:.3f} ms")


Python: 3.13.15
Synthetic samples: 5
Unicode inspection=PASS
Two-copy preprocessing contract=PASS
Known abbreviation probe: ['راجع د.', 'أحمد.', 'ثم أعد المحاولة.']
SPACY_SENTENCE_PIPELINE=PASS
Local WordPiece demonstration=PASS
Local mean token fertility: 1.44
Local truncation rate @10: 20%
Tokenisation metrics=PASS
input_ids shape: (2, 12)
attention_mask shape: (2, 12)
embeddings shape: (2, 12, 8)
IDs-to-embeddings pipeline=PASS


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]


=== Explore: Local tokenizer vs mBERT ===
Sample 1: local=10 tokens, mBERT=17 tokens, local_UNK=0
Sample 2: local=9 tokens, mBERT=13 tokens, local_UNK=4
Sample 3: local=11 tokens, mBERT=14 tokens, local_UNK=8
Sample 4: local=9 tokens, mBERT=7 tokens, local_UNK=0
Sample 5: local=13 tokens, mBERT=11 tokens, local_UNK=9
EXPLORE_TOKENIZER_COMPARISON=PASS
mBERT: fertility=1.72, truncation@10=80%, avg_time=0.076 ms
XLM-R: fertility=1.43, truncation@10=80%, avg_time=0.086 ms
DISTINCTION_TOKENIZER_COMPARISON=PASS
unicode: PASS
spacy_sentence_pipeline: PASS
raw_copy_preserved: PASS
pii_masked: PASS
token_metrics: PASS
embedding_shape: PASS
DAY1_NOTEBOOK1_CORE=PASS

=== DAY 1 EVIDENCE SUMMARY ===
Profile: Conservative (NFC + whitespace normalization + tatweel removal + PII masking)
PII example: 0558764321 -> <PHONE>
Local mean token fertility: 1.44
Local truncation rate @10: 20%
Decision: Preserve Arabic text with conservative normalization; aggressive normalization is not used by default.
mBER